# train

> Training loop, run history, policy evaluation, and multi-run helpers.


In [ ]:
#| default_exp train


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from dataclasses import dataclass
import numpy as np


In [ ]:
#| export
@dataclass
class History:
    t: np.ndarray
    theta: np.ndarray
    saccade: np.ndarray
    dt: np.ndarray
    reward: np.ndarray


In [ ]:
#| export
def train_agent(agent, env, Ttot=None):
    'Run Algorithm 1 for Ttot seconds of simulated time.'
    if Ttot is None:
        Ttot = agent.params.Ttot
    p = agent.params
    n_steps = round(Ttot / p.delT)
    t_arr = np.zeros(n_steps, dtype=np.float64)
    theta_arr = np.zeros(n_steps, dtype=np.float64)
    saccade_arr = np.zeros(n_steps, dtype=bool)
    dt_arr = np.zeros(n_steps, dtype=np.int64)
    reward_arr = np.zeros(n_steps, dtype=np.float64)
    t, i = 0, 0
    while t < n_steps:
        step = agent.rl.chooseAction(agent)
        new_theta, r = env.step(agent.theta, step.dx)
        agent.rl.updateParams(agent, step, r)
        t_arr[i] = t
        theta_arr[i] = step.theta
        saccade_arr[i] = step.saccade
        dt_arr[i] = step.dt
        reward_arr[i] = r
        agent.theta = new_theta
        t += step.dt
        i += 1
    return History(t_arr[:i], theta_arr[:i], saccade_arr[:i], dt_arr[:i], reward_arr[:i])


In [ ]:
#| export
def evaluatePolicy(agent):
    'Drift rate, turn probability, fixation duration, and turn bias at each heading column.'
    pol = agent.policy
    p = agent.params
    nx = p.nx
    nu = np.empty(nx)
    pR = np.empty(nx)
    for j in range(nx):
        nu[j], pR[j] = agent.convertParams(pol.omega, pol.fk[:, j], pol.wParams)
    return dict(nu=nu, pR=pR, fixDur=p.aF / nu, turnBias=2 * pR - 1)


In [ ]:
#| export
def residency(history, env, nBins=8, halfWidthDeg=45):
    'Fraction of time within halfWidthDeg of the goal in each equal time bin.'
    res = np.zeros(nBins)
    if len(history.t) == 0:
        return res
    half_w = np.deg2rad(halfWidthDeg)
    t_end = float(history.t[-1] + history.dt[-1])
    edges = np.linspace(0, t_end, nBins + 1)
    near = np.zeros(nBins)
    w = np.zeros(nBins)
    for ti, th, dt in zip(history.t, history.theta, history.dt):
        t0, t1 = float(ti), float(ti + dt)
        for k in range(nBins):
            lo, hi = edges[k], edges[k + 1]
            overlap = max(0.0, min(t1, hi) - max(t0, lo))
            if overlap <= 0:
                continue
            w[k] += overlap
            if env.distance(th) <= half_w:
                near[k] += overlap
    for k in range(nBins):
        res[k] = near[k] / w[k] if w[k] > 0 else 0.0
    return res


In [ ]:
#| export
def runMany(params, goal, nRuns, seed=0, Ttot=None):
    'Independent training runs; each result has eval, residency, and final omega.'
    from explRL.core import loadAgent, loadEnv
    from explRL.params import Params
    if params is None:
        params = Params()
    out = []
    for i in range(nRuns):
        agent = loadAgent(params=params, goal=goal, seed=seed + i)
        env = loadEnv(params=params, goal=goal)
        hist = train_agent(agent, env, Ttot=Ttot)
        out.append({
            'eval': evaluatePolicy(agent),
            'residency': residency(hist, env),
            'omega': agent.policy.omega.copy(),
        })
        del hist
    return out


In [ ]:
#| export
import json
from pathlib import Path


def demo_artifact_dir(mode='quick', name='demo'):
    'Directory ``results/demo/{mode}/`` (or ``results/demo`` when ``mode`` is None).'
    root = Path.cwd()
    while not (root / 'pyproject.toml').exists() and root != root.parent:
        root = root.parent
    base = root / 'results' / name
    if mode is None:
        base.mkdir(parents=True, exist_ok=True)
        return base
    d = base / mode
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_demo_figure(name, fig=None, mode='quick', demo_name='demo', **kwargs):
    'Save a matplotlib figure as ``{name}.png`` under the demo results directory.'
    import matplotlib.pyplot as plt
    fig = fig or plt.gcf()
    path = demo_artifact_dir(mode, name=demo_name) / f'{name}.png'
    fig.savefig(path, bbox_inches='tight', dpi=150, **kwargs)
    return path


def save_demo_metrics(metrics, mode='full', name='metrics.json', demo_name='demo'):
    'Write demo acceptance metrics and run metadata as JSON.'
    path = demo_artifact_dir(mode, name=demo_name) / name
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
        f.write('\n')
    return path

In [ ]:
from math import pi
from fastcore.test import test_eq, test_close
from explRL.params import Params
from explRL.core import loadAgent, loadEnv

p = Params(aF=0.5)
agent = loadAgent(params=p, seed=1)
env = loadEnv(goal=pi)
hist = train_agent(agent, env, Ttot=2.0)
assert len(hist.t) > 0
assert hist.saccade.any()
target = round(2.0 / p.delT)
assert int(hist.dt.sum()) >= target
assert int(hist.dt.sum()) <= target + p.dtS
assert np.isfinite(agent.policy.omega).all()


In [ ]:
h0 = train_agent(loadAgent(params=p, seed=7), env, Ttot=2.0)
h1 = train_agent(loadAgent(params=p, seed=7), env, Ttot=2.0)
for a, b in zip(h0.t, h1.t):
    test_eq(a, b)
for a, b in zip(h0.theta, h1.theta):
    test_eq(a, b)
test_eq(h0.saccade.tolist(), h1.saccade.tolist())
test_eq(h0.dt.tolist(), h1.dt.tolist())
test_close(h0.reward, h1.reward)
h2 = train_agent(loadAgent(params=p, seed=8), env, Ttot=2.0)
assert len(h2.t) != len(h0.t) or not np.allclose(h2.theta, h0.theta)


In [ ]:
agent = loadAgent(seed=0)
ev = evaluatePolicy(agent)
test_eq(len(ev['nu']), p.nx)
test_eq(len(ev['pR']), p.nx)
assert (ev['fixDur'] > 0).all()
assert ev['turnBias'].min() >= -1 and ev['turnBias'].max() <= 1
z = loadAgent(params=Params(omegaS0=0.0), seed=0)
evz = evaluatePolicy(z)
test_close(evz['turnBias'], evz['turnBias'][0])


In [ ]:
env = loadEnv(goal=pi)
at_goal = History(
    t=np.array([0.0, 100.0]),
    theta=np.array([pi, pi]),
    saccade=np.array([False, False]),
    dt=np.array([100, 100], dtype=np.int64),
    reward=np.array([0.0, 0.0]),
)
test_close(residency(at_goal, env, nBins=4), np.ones(4))


In [ ]:
mixed = History(
    t=np.array([0.0, 1.0]),
    theta=np.array([0.0, pi]),
    saccade=np.array([False, True]),
    dt=np.array([1, p.dtS], dtype=np.int64),
    reward=np.array([0.0, 0.0]),
)
r1 = residency(mixed, env, nBins=1, halfWidthDeg=90)
w_fix, w_sac = 1.0, float(p.dtS)
test_close(r1[0], w_sac / (w_fix + w_sac))


In [ ]:
runs = runMany(params=p, goal=pi, nRuns=2, seed=10, Ttot=1.0)
test_eq(len(runs), 2)
for d in runs:
    test_eq(set(d.keys()), {'eval', 'residency', 'omega'})
    test_eq(len(d['eval']['nu']), p.nx)


In [ ]:
from fastcore.test import test_eq

d = demo_artifact_dir('quick')
test_eq(d.name, 'quick')
test_eq(d.parent.name, 'demo')
test_eq(demo_artifact_dir(None).name, 'demo')
d2 = demo_artifact_dir('quick', name='demo_prestructured')
test_eq(d2.parent.name, 'demo_prestructured')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
